# Part 3 - Data Preprocessing
## Creating Congestion Categories and Proxy Accident-Risk Labels

**This notebook continues from Part 2.**

Loads the cleaned dataset from Part 2 and adds Part 3-specific features:
1. Congestion categories (based on traffic volume quartiles)
2. Proxy accident-risk labels (high congestion + severe weather/low visibility)

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported")

✓ Libraries imported


## 2. Load Cleaned Data from Part 2

In [2]:
# Load the cleaned dataset from Part 2
df = pd.read_csv('../../Part 2/Metro_Interstate_Traffic_Volume_cleaned.csv')

print(f"✓ Cleaned data loaded from Part 2")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")
print(f"\nData Info:")
print(df.info())

✓ Cleaned data loaded from Part 2
  Shape: (48187, 9)
  Columns: ['holiday', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'date_time', 'traffic_volume']

Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48187 entries, 0 to 48186
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   holiday              61 non-null     object 
 1   temp                 48187 non-null  float64
 2   rain_1h              48187 non-null  float64
 3   snow_1h              48187 non-null  float64
 4   clouds_all           48187 non-null  int64  
 5   weather_main         48187 non-null  object 
 6   weather_description  48187 non-null  object 
 7   date_time            48187 non-null  object 
 8   traffic_volume       48187 non-null  int64  
dtypes: float64(3), int64(2), object(4)
memory usage: 3.3+ MB
None


## 3. Create Congestion Categories

Categorize traffic volume into 4 levels based on quartiles:
- **Low**: ≤ Q1 (25th percentile)
- **Medium**: Q1 to Q2 (25th-50th percentile)  
- **High**: Q2 to Q3 (50th-75th percentile)
- **Severe**: > Q3 (75th percentile)

In [3]:
# Calculate traffic volume quartiles
q1, q2, q3 = df["traffic_volume"].quantile([0.25, 0.5, 0.75]).values

print("Traffic Volume Quartiles:")
print(f"  Q1 (25%): {q1:.2f} vehicles/hour")
print(f"  Q2 (50%): {q2:.2f} vehicles/hour (Median)")
print(f"  Q3 (75%): {q3:.2f} vehicles/hour")

# Define congestion categorization function
def categorize_congestion(volume):
    if volume <= q1:
        return "Low"
    elif volume <= q2:
        return "Medium"
    elif volume <= q3:
        return "High"
    else:
        return "Severe"

# Apply categorization
df["congestion_category"] = df["traffic_volume"].apply(categorize_congestion)

print(f"\nCongestion Category Distribution:")
print(df["congestion_category"].value_counts().sort_index())
print(f"\nPercentage Distribution:")
print((df["congestion_category"].value_counts(normalize=True).sort_index() * 100).round(2))

Traffic Volume Quartiles:
  Q1 (25%): 1192.50 vehicles/hour
  Q2 (50%): 3379.00 vehicles/hour (Median)
  Q3 (75%): 4933.00 vehicles/hour

Congestion Category Distribution:
congestion_category
High      12054
Low       12047
Medium    12049
Severe    12037
Name: count, dtype: int64

Percentage Distribution:
congestion_category
High      25.02
Low       25.00
Medium    25.00
Severe    24.98
Name: proportion, dtype: float64


## 4. Create Proxy Accident-Risk Label

**High-Risk Definition**: High or Severe congestion occurring together with:
- Severe weather (Thunderstorm, Snow, Fog, Mist, Smoke, Haze, Dust, Ash), OR
- Low visibility (cloudiness > 75% or fog/mist in description)

In [4]:
# Define severe weather types
SEVERE_WEATHER = ["Thunderstorm", "Snow", "Fog", "Mist", "Smoke", "Haze", "Dust", "Ash"]

print("Severe Weather Types:")
print(f"  {SEVERE_WEATHER}")

# Create low visibility indicator
df["is_low_visibility"] = (
    (df["clouds_all"] > 75) | 
    df["weather_description"].str.contains("fog|mist", case=False, na=False).astype(int)
).astype(int)

print(f"\nLow Visibility Records: {df['is_low_visibility'].sum():,} ({df['is_low_visibility'].mean()*100:.2f}%)")

Severe Weather Types:
  ['Thunderstorm', 'Snow', 'Fog', 'Mist', 'Smoke', 'Haze', 'Dust', 'Ash']

Low Visibility Records: 20,786 (43.14%)


In [5]:
# Create high-risk label
high_congestion = df["congestion_category"].isin(["High", "Severe"])
risky_weather = (
    df["weather_main"].isin(SEVERE_WEATHER) | 
    (df["is_low_visibility"] == 1)
)

df["high_risk"] = (high_congestion & risky_weather).astype(int)

print("High-Risk Label Distribution:")
risk_dist = df["high_risk"].value_counts().sort_index()
for idx, count in risk_dist.items():
    label = "High Risk" if idx == 1 else "No Risk"
    pct = count / len(df) * 100
    print(f"  {label}: {count:,} ({pct:.2f}%)")

print(f"\nRisk Factor Analysis:")
print(f"  High/Severe Congestion: {high_congestion.sum():,} records")
print(f"  Risky Weather Conditions: {risky_weather.sum():,} records")
print(f"  High Risk (Both conditions): {df['high_risk'].sum():,} records")

High-Risk Label Distribution:
  No Risk: 37,242 (77.29%)
  High Risk: 10,945 (22.71%)

Risk Factor Analysis:
  High/Severe Congestion: 24,091 records
  Risky Weather Conditions: 22,604 records
  High Risk (Both conditions): 10,945 records


In [6]:
# Save dataset with new Part 3 features
df.to_csv('Metro_Interstate_Traffic_Volume_part3_preprocessed.csv', index=False)
print("✓ Part 3 preprocessed data saved")
print(f"  File: Metro_Interstate_Traffic_Volume_part3_preprocessed.csv")
print(f"  Shape: {df.shape}")

# Save quartile information for reference
import pickle

part3_metadata = {
    "q1": q1,
    "q2": q2,
    "q3": q3,
    "SEVERE_WEATHER": SEVERE_WEATHER,
    "description": "Part 3 preprocessing metadata - congestion quartiles and severe weather types"
}

with open('part3_metadata.pkl', 'wb') as f:
    pickle.dump(part3_metadata, f)

print(f"✓ Part 3 metadata saved")
print(f"  File: part3_metadata.pkl")

print(f"\nDataset Summary:")
print(f"  Total rows: {len(df):,}")
print(f"  Total columns: {df.shape[1]}")
print(f"  New columns added: congestion_category, high_risk, is_low_visibility")

✓ Part 3 preprocessed data saved
  File: Metro_Interstate_Traffic_Volume_part3_preprocessed.csv
  Shape: (48187, 12)
✓ Part 3 metadata saved
  File: part3_metadata.pkl

Dataset Summary:
  Total rows: 48,187
  Total columns: 12
  New columns added: congestion_category, high_risk, is_low_visibility
